## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-22-10-54-59 +0000,nyt,How a Crash Into a Cruise Ship in New Orleans ...,https://www.nytimes.com/2026/11/22/us/new-orle...
1,2026-11-22-10-53-19 +0000,nyt,A Riverboat Captain Recalls a Crash on the Mis...,https://www.nytimes.com/2026/11/22/us/new-orle...
2,2026-11-22-10-44-46 +0000,bbc,Theresa May 'aghast' at Burnham's EU plans and...,https://www.bbc.co.uk/news/articles/cvj6l8y3x1...
3,2026-11-22-10-30-53 +0000,nypost,‘Socialist’ AOC spent $12K on glam German digs...,https://nypost.com/2026/11/22/us-news/aoc-spen...
4,2026-11-22-10-30-00 +0000,wsj,James Bond’s Bare Wrist Sets Off Watch Industr...,https://www.wsj.com/business/retail/james-bond...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2504/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
92,trump,21
3,new,8
175,dead,7
171,fire,7
70,house,7


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
139,2026-11-21-16-32-51 +0000,nypost,Trump defends Stephen Miller after Hakeem Jeff...,https://nypost.com/2026/11/21/us-news/trump-de...,55
144,2026-11-21-16-11-17 +0000,nypost,Trump thinks there may be upside to reported c...,https://nypost.com/2026/11/21/us-news/trump-ta...,54
38,2026-11-22-06-43-17 +0000,nypost,Mike Waltz warns Russia faces ‘unsustainable’ ...,https://nypost.com/2026/11/22/us-news/russia-f...,45
159,2026-11-21-14-40-55 +0000,nypost,Two tankers hit in Strait of Hormuz as hawks p...,https://nypost.com/2026/11/21/world-news/two-t...,44
152,2026-11-21-15-10-50 +0000,chitri,Mexico detains top cartel boss ‘El Chapo Isidr...,https://www.chicagotribune.com/2026/11/21/el-c...,41


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
139,55,2026-11-21-16-32-51 +0000,nypost,Trump defends Stephen Miller after Hakeem Jeff...,https://nypost.com/2026/11/21/us-news/trump-de...
152,41,2026-11-21-15-10-50 +0000,chitri,Mexico detains top cartel boss ‘El Chapo Isidr...,https://www.chicagotribune.com/2026/11/21/el-c...
72,37,2026-11-22-00-17-14 +0000,nypost,Maniac accused of killing NYC woman in possibl...,https://nypost.com/2026/11/21/us-news/suspecte...
19,37,2026-11-22-10-00-00 +0000,wsj,Richer and more adventurous than their predece...,https://www.wsj.com/personal-finance/retiremen...
29,34,2026-11-22-08-43-29 +0000,nypost,Ex-White House press secretary Katie Zacharia ...,https://nypost.com/2026/11/22/us-news/ex-white...
49,31,2026-11-22-04-05-27 +0000,nypost,ICE officers arrest Alaska Airlines passengers...,https://nypost.com/2026/11/21/us-news/masked-i...
116,31,2026-11-21-19-13-56 +0000,nypost,"2 dead, 60 injured in Russian strike on Zelens...",https://nypost.com/2026/11/21/world-news/2-dea...
81,29,2026-11-21-23-50-12 +0000,chitri,Nations agree to extend talks on AI killer rob...,https://www.chicagotribune.com/2026/11/21/nati...
144,27,2026-11-21-16-11-17 +0000,nypost,Trump thinks there may be upside to reported c...,https://nypost.com/2026/11/21/us-news/trump-ta...
107,25,2026-11-21-19-51-37 +0000,chitri,Iran strikes 3 tankers as it shows off missile...,https://www.chicagotribune.com/2026/11/21/iran...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
